# Stage 2 - twin fitting, leave-one-community-out, conformal, E1 (CPU runtime)

Produces results.json and report.md. Before the real run, run `python -m pytest tests` once so you know the code works in your Colab.

In [ ]:
# Kaggle setup. Settings panel: Internet ON for stage 0 only; GPU only for stage 1.
# Add your private dataset "stresstwin-p" via Add Input. The folder that contains pyproject.toml is found automatically.
import os, shutil, subprocess, sys
def find_pkg():
    """Find the package wherever Kaggle mounted the dataset (the mount path differs between Kaggle versions), or unzip it if it was left zipped."""
    import zipfile
    for r, dirs, files in os.walk('/kaggle/input'):
        if 'pyproject.toml' in files and os.path.isdir(os.path.join(r, 'stresstwin_p')):
            return r
    for r, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.zip') and 'stresstwin' in f.lower() and 'reproducibility' not in f.lower():
                out = '/kaggle/working/pkg_unzipped'; zipfile.ZipFile(os.path.join(r, f)).extractall(out)
                for r2, d2, f2 in os.walk(out):
                    if 'pyproject.toml' in f2 and os.path.isdir(os.path.join(r2, 'stresstwin_p')):
                        return r2
    print('Nothing found. What is mounted under /kaggle/input:')
    for r, d, f in os.walk('/kaggle/input'):
        if r.count(os.sep) <= 5: print(' ', r, '| files:', f[:6])
    return None
root = find_pkg()
assert root, 'package not found: in the right panel click Add Input -> Your Datasets -> stresstwin-p (it must be listed under Input), then run again'
if not os.path.exists('/kaggle/working/stresstwin_pkg'): shutil.copytree(root, '/kaggle/working/stresstwin_pkg')
subprocess.run([sys.executable,'-m','pip','-q','install','-e','/kaggle/working/stresstwin_pkg'],check=True)
sys.path.insert(0, '/kaggle/working/stresstwin_pkg')
WORK='/kaggle/working/work'; os.makedirs(WORK, exist_ok=True)
# /kaggle/working is kept only as notebook OUTPUT. After each stage: Save Version, then make the output a PRIVATE dataset.
# Never make a dataset containing texts.csv public.
def find_input(name, kind='file'):
    """Locate a file or folder by name anywhere under /kaggle/input (the mount path differs between Kaggle versions)."""
    for r, dirs, files in os.walk('/kaggle/input'):
        if kind == 'file' and name in files: return os.path.join(r, name)
        if kind == 'dir' and name in dirs: return os.path.join(r, name)
    raise FileNotFoundError(f'{name} not found under /kaggle/input. Add the dataset that holds it (right panel: Add Input).')


In [ ]:
# Confirmatory run: all authors EXCEPT the pilot slice. Set PILOT_MODE = 'only' to reproduce the pilot instead.
import subprocess, sys
PILOT_MODE = 'exclude'
SEEDS = [0, 1, 2, 3, 4]            # author-partition seeds for the robustness table
obs_f, thr_f = find_input('observations.csv'), find_input('threads.csv')
sc_f = find_input('scores_stresstwin_f.csv')
try: plain_f = find_input('scores_plain_ft.csv')
except FileNotFoundError: plain_f = None; print('plain-scorer scores not found: the scorer ablation (A1) will be skipped')
for seed in SEEDS:
    cmd = [sys.executable, '-m', 'stresstwin_p.run_all', '--obs', obs_f, '--scores', sc_f, '--threads', thr_f,
           '--out', f'{WORK}/results_seed{seed}', '--seed', str(seed), '--pilot', PILOT_MODE]
    if plain_f: cmd += ['--scores-plain', plain_f]
    print('$', ' '.join(cmd)); subprocess.run(cmd, check=True)

## How to read it (decide BEFORE looking)
- Twin must beat `author_mean`, `ewma`, `gap_ewma` on **both** MSE and NLL in held-out communities, otherwise the person-level claim is not supported.
- `twin[split_half]` vs `twin[conf_rule]` vs `twin[fixed]` is the noise-model comparison (the AI-YOU-style rule vs measured noise).
- Static conformal should lose coverage on held-out communities; ACI should repair it. If neither happens, report that.
- E1 Gate A: beta<0, p<0.05, |placebo|<0.5|beta|. Failing it is a reportable null, not something to tune around.